In [1]:
import pandas as pd
import json
from pathlib import Path


In [2]:
FILES = {
    'RH':  'RH_Talan_Tunisie.xlsx',
    'CRM': 'CRM_Talan_Tunisie.xlsx',
    'ERP': 'ERP_Talan_Tunisie.xlsx',
}

In [3]:
def load(fname, sheet):
    df = pd.read_excel(fname, sheet_name=sheet)
    df.columns = df.iloc[0]
    return df.iloc[1:].reset_index(drop=True)


In [4]:
# ─────────────────────────────────────────────
# 1. Lire toutes les tables (hors Résumé)
# ─────────────────────────────────────────────
tables = {}
for sys_name, fname in FILES.items():
    xl = pd.ExcelFile(fname)
    for sheet in xl.sheet_names:
        if 'résumé' in sheet.lower() or 'resume' in sheet.lower() or '📋' in sheet:
            continue
        df = load(fname, sheet)
        key = f"{sys_name}/{sheet}"
        tables[key] = df
        print(f"  Chargé : {key:45s} → {len(df):>5} lignes  ×  {len(df.columns):>2} colonnes")


  Chargé : RH/departments                                →    10 lignes  ×   6 colonnes
  Chargé : RH/employees                                  →   150 lignes  ×  15 colonnes
  Chargé : RH/projects                                   →    60 lignes  ×  11 colonnes
  Chargé : RH/employee_projects                          →   300 lignes  ×   7 colonnes
  Chargé : RH/leave_requests                             →   200 lignes  ×  10 colonnes
  Chargé : RH/skills                                     →   606 lignes  ×   7 colonnes
  Chargé : RH/performance_reviews                        →   348 lignes  ×  13 colonnes
  Chargé : RH/timesheets                                 →   887 lignes  ×  12 colonnes
  Chargé : RH/project_milestones                         →   480 lignes  ×  10 colonnes
  Chargé : RH/recruitment_pipeline                       →    80 lignes  ×  14 colonnes
  Chargé : CRM/accounts                                  →   200 lignes  ×  10 colonnes
  Chargé : CRM/contacts         

In [5]:
# ─────────────────────────────────────────────
# 2. Analyse NULLs
# ─────────────────────────────────────────────
print("\n── NULLS DÉTECTÉS ──")
for key, df in tables.items():
    nulls = df.isnull().sum()
    null_cols = nulls[nulls > 0]
    if len(null_cols):
        for col, n in null_cols.items():
            pct = round(n / len(df) * 100, 1)
            print(f"  {key}.{col}: {n} NULLs ({pct}%)")


── NULLS DÉTECTÉS ──
  RH/employees.manager_id: 20 NULLs (13.3%)
  RH/leave_requests.notes: 80 NULLs (40.0%)
  RH/timesheets.notes: 438 NULLs (49.4%)
  RH/project_milestones.actual_cost: 142 NULLs (29.6%)
  RH/project_milestones.notes: 248 NULLs (51.7%)
  RH/recruitment_pipeline.actual_hire_date: 70 NULLs (87.5%)
  CRM/leads.lost_reason: 472 NULLs (78.7%)
  CRM/opportunities.lost_reason: 417 NULLs (83.4%)
  ERP/sales_orders.notes: 210 NULLs (21.0%)
  ERP/po_lines.notes: 755 NULLs (62.4%)


| Colonne                        | NULL signifie      |
| ------------------------------ | ------------------ | 
| `employees.manager_id`         | Top manager        |       
| `*.notes`                      | Pas de commentaire |  
| `milestones.actual_cost`       | Jalon pas terminé  | 
| `recruitment.actual_hire_date` | Poste pas pourvu   |
| `*.lost_reason`                | Deal pas perdu     |


In [9]:
# ─────────────────────────────────────────────
# 3. Validation FK cross-système
# ─────────────────────────────────────────────
print("\n── FK CROSS-SYSTÈME ──")

emp_ids  = set(tables['RH/employees']['employee_id'].dropna())
acc_ids  = set(tables['CRM/accounts']['account_id'].dropna())
cust_ids = set(tables['ERP/customers']['customer_id'].dropna())
sup_ids  = set(tables['ERP/suppliers']['supplier_id'].dropna())
proj_ids = set(tables['RH/projects']['project_id'].dropna())
prod_ids = set(tables['ERP/products']['product_id'].dropna())
ord_ids  = set(tables['ERP/sales_orders']['order_id'].dropna())
inv_ids  = set(tables['ERP/invoices']['invoice_id'].dropna())
po_ids   = set(tables['ERP/purchase_orders']['po_id'].dropna())
cross_fk_checks = [
    ("ERP/customers",          "account_id",         acc_ids,  "CRM/accounts.account_id"),
    ("RH/projects",            "client_account_id",  acc_ids,  "CRM/accounts.account_id"),
    ("CRM/opportunities",      "owner_id",           emp_ids,  "RH/employees.employee_id"),
    ("ERP/sales_orders",       "sales_rep_id",       emp_ids,  "RH/employees.employee_id"),
    ("ERP/purchase_orders",    "approved_by",        emp_ids,  "RH/employees.employee_id"),
    ("RH/departments",         "manager_id",         emp_ids,  "RH/employees.employee_id (auto-ref)"),
    ("ERP/products",           "supplier_id",        sup_ids,  "ERP/suppliers.supplier_id"),
    ("ERP/sales_orders",       "customer_id",        cust_ids, "ERP/customers.customer_id"),
    ("ERP/invoices",           "customer_id",        cust_ids, "ERP/customers.customer_id"),
    ("ERP/invoices",           "order_id",           ord_ids,  "ERP/sales_orders.order_id"),
    ("ERP/order_lines",        "order_id",           ord_ids,  "ERP/sales_orders.order_id"),
    ("ERP/order_lines",        "product_id",         prod_ids, "ERP/products.product_id"),
    ("ERP/payments",           "invoice_id",         inv_ids,  "ERP/invoices.invoice_id"),
    ("ERP/po_lines",           "po_id",              po_ids,   "ERP/purchase_orders.po_id"),
    ("ERP/po_lines",           "product_id",         prod_ids, "ERP/products.product_id"),
    ("RH/employee_projects",   "employee_id",        emp_ids,  "RH/employees.employee_id"),
    ("RH/employee_projects",   "project_id",         proj_ids, "RH/projects.project_id"),
    ("RH/timesheets",          "employee_id",        emp_ids,  "RH/employees.employee_id"),
    ("RH/timesheets",          "project_id",         proj_ids, "RH/projects.project_id"),
    ("RH/skills",              "employee_id",        emp_ids,  "RH/employees.employee_id"),
    ("RH/leave_requests",      "employee_id",        emp_ids,  "RH/employees.employee_id"),
    ("RH/performance_reviews", "employee_id",        emp_ids,  "RH/employees.employee_id"),
]

results = []
for table_key, col, ref_set, ref_label in cross_fk_checks:
    df = tables[table_key]
    if col not in df.columns:
        continue
    vals = set(df[col].dropna())
    n_total = len(vals)
    orphans = vals - ref_set
    status = "✅ OK" if len(orphans) == 0 else f"⚠️  {len(orphans)} orphelins"
    print(f"  {table_key}.{col} → {ref_label}: {n_total} refs  {status}")
    results.append({
        'from': f"{table_key}.{col}",
        'to': ref_label,
        'refs': n_total,
        'orphans': len(orphans),
    })



── FK CROSS-SYSTÈME ──
  ERP/customers.account_id → CRM/accounts.account_id: 200 refs  ✅ OK
  RH/projects.client_account_id → CRM/accounts.account_id: 55 refs  ✅ OK
  CRM/opportunities.owner_id → RH/employees.employee_id: 144 refs  ✅ OK
  ERP/sales_orders.sales_rep_id → RH/employees.employee_id: 150 refs  ✅ OK
  ERP/purchase_orders.approved_by → RH/employees.employee_id: 146 refs  ✅ OK
  RH/departments.manager_id → RH/employees.employee_id (auto-ref): 9 refs  ✅ OK
  ERP/products.supplier_id → ERP/suppliers.supplier_id: 118 refs  ✅ OK
  ERP/sales_orders.customer_id → ERP/customers.customer_id: 200 refs  ✅ OK
  ERP/invoices.customer_id → ERP/customers.customer_id: 193 refs  ✅ OK
  ERP/invoices.order_id → ERP/sales_orders.order_id: 611 refs  ✅ OK
  ERP/order_lines.order_id → ERP/sales_orders.order_id: 1000 refs  ✅ OK
  ERP/order_lines.product_id → ERP/products.product_id: 495 refs  ✅ OK
  ERP/payments.invoice_id → ERP/invoices.invoice_id: 526 refs  ✅ OK
  ERP/po_lines.po_id → ERP/purchas

In [8]:

# ─────────────────────────────────────────────
# 4. Cardinalités
# ─────────────────────────────────────────────
print("\n── CARDINALITÉS ──")

cardinalities = [
    ("RH/departments",         "department_id",  "RH/employees",         "department_id",  "1 → N"),
    ("RH/employees",           "employee_id",    "RH/skills",            "employee_id",    "1 → N"),
    ("RH/employees",           "employee_id",    "RH/leave_requests",    "employee_id",    "1 → N"),
    ("RH/employees",           "employee_id",    "RH/performance_reviews","employee_id",   "1 → N"),
    ("RH/employees",           "employee_id",    "RH/timesheets",        "employee_id",    "1 → N"),
    ("RH/employees",           "employee_id",    "RH/employee_projects", "employee_id",    "N ↔ N"),
    ("RH/projects",            "project_id",     "RH/employee_projects", "project_id",     "N ↔ N"),
    ("RH/projects",            "project_id",     "RH/project_milestones","project_id",     "1 → N"),
    ("CRM/accounts",           "account_id",     "CRM/contacts",         "account_id",     "1 → N"),
    ("CRM/accounts",           "account_id",     "CRM/opportunities",    "account_id",     "1 → N"),
    ("CRM/accounts",           "account_id",     "CRM/revenue_history",  "account_id",     "1 → N"),
    ("CRM/contacts",           "contact_id",     "CRM/activities",       "contact_id",     "1 → N"),
    ("ERP/customers",          "customer_id",    "ERP/sales_orders",     "customer_id",    "1 → N"),
    ("ERP/sales_orders",       "order_id",       "ERP/order_lines",      "order_id",       "1 → N"),
    ("ERP/sales_orders",       "order_id",       "ERP/invoices",         "order_id",       "1 → N"),
    ("ERP/suppliers",          "supplier_id",    "ERP/products",         "supplier_id",    "1 → N"),
    ("ERP/suppliers",          "supplier_id",    "ERP/purchase_orders",  "supplier_id",    "1 → N"),
    ("ERP/purchase_orders",    "po_id",          "ERP/po_lines",         "po_id",          "1 → N"),
    ("ERP/invoices",           "invoice_id",     "ERP/payments",         "invoice_id",     "1 → N"),
    ("ERP/products",           "product_id",     "ERP/inventory",        "product_id",     "1 → N"),
]

for parent_key, pk_col, child_key, fk_col, card in cardinalities:
    parent = tables.get(parent_key)
    child  = tables.get(child_key)
    if parent is None or child is None:
        continue
    n_parent = parent[pk_col].nunique()
    grp = child.groupby(fk_col).size()
    avg = grp.mean()
    mn  = grp.min()
    mx  = grp.max()
    print(f"  {parent_key}.{pk_col} {card} {child_key}.{fk_col}  |  {n_parent} parents → avg {avg:.1f}, min {mn}, max {mx} enfants")

print("\nAudit terminé.")


── CARDINALITÉS ──
  RH/departments.department_id 1 → N RH/employees.department_id  |  10 parents → avg 15.0, min 10, max 21 enfants
  RH/employees.employee_id 1 → N RH/skills.employee_id  |  150 parents → avg 4.0, min 3, max 6 enfants
  RH/employees.employee_id 1 → N RH/leave_requests.employee_id  |  150 parents → avg 1.9, min 1, max 5 enfants
  RH/employees.employee_id 1 → N RH/performance_reviews.employee_id  |  150 parents → avg 2.3, min 1, max 4 enfants
  RH/employees.employee_id 1 → N RH/timesheets.employee_id  |  150 parents → avg 6.6, min 3, max 18 enfants
  RH/employees.employee_id N ↔ N RH/employee_projects.employee_id  |  150 parents → avg 2.2, min 1, max 6 enfants
  RH/projects.project_id N ↔ N RH/employee_projects.project_id  |  60 parents → avg 5.0, min 1, max 12 enfants
  RH/projects.project_id 1 → N RH/project_milestones.project_id  |  60 parents → avg 8.0, min 8, max 8 enfants
  CRM/accounts.account_id 1 → N CRM/contacts.account_id  |  200 parents → avg 4.1, min 1, ma